In [ ]:
from pathlib import Path
import os

# Colab에서 PALSYN 루트로 이동
PALSYN_DIR = Path("/content/drive/MyDrive/PM_Assessment_Logs/PALSYN")
os.chdir(PALSYN_DIR)

DATA_PATH = PALSYN_DIR / "data" / "MIMICEL" / "mimicel_train.xes"
MODEL_DIR = PALSYN_DIR / "models" / "mimicel_train"
RESULT_DIR = PALSYN_DIR / "results" / "mimicel_train"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("cwd:", os.getcwd())
print("data:", DATA_PATH.exists(), DATA_PATH)

In [ ]:
import pm4py
from PALSYN.synthesizer import LSTMSynthesizer

print("PALSYN LSTM import OK")

In [ ]:
event_log = pm4py.read_xes(str(DATA_PATH))

print("trace count:", len(event_log))

activities = sorted(set(
    event["concept:name"]
    for trace in event_log
    for event in trace
))

print("activity count:", len(activities))
print(activities)

In [ ]:
small_log = event_log[:100]

palsyn_model = LSTMSynthesizer(
    pre_processing={
        "max_clusters": 10,
        "trace_quantile": 0.9,
        "seed": 88,
    },
    model={
        "embedding_output_dims": 64,
        "epochs": 3,
        "batch_size": 32,
        "validation_split": 0.15,
        "units_per_layer": [32, 16],
        "dropout": 0.0,
        "bidirectional": True,
    },
    dp_optimizer={
        "epsilon": 15.0,
        "learning_rate": 5e-4,
        "l2_norm_clip": 1.0,
    },
)

palsyn_model.fit(small_log)

In [ ]:
palsyn_model.save_model(str(MODEL_DIR))

print("saved model:", MODEL_DIR)

In [ ]:
synthetic_log = palsyn_model.sample(
    sample_size=100,
    batch_size=20
)

print("synthetic traces:", len(synthetic_log))

In [ ]:
from PALSYN.postprocessing.log_postprocessing import clean_xes_file

synthetic_event_log = pm4py.convert_to_event_log(synthetic_log)

OUT_XES = RESULT_DIR / "palsyn_synthetic_test.xes"
OUT_CSV = RESULT_DIR / "palsyn_synthetic_test.csv"

pm4py.write_xes(synthetic_event_log, str(OUT_XES))
clean_xes_file(str(OUT_XES), str(OUT_XES))

df_syn = pm4py.convert_to_dataframe(synthetic_event_log)
df_syn["time:timestamp"] = df_syn["time:timestamp"].astype(str)
df_syn.to_csv(OUT_CSV, index=False)

print("saved xes:", OUT_XES)
print("saved csv:", OUT_CSV)
print(df_syn.head())